# FantasAI Weekly Stats Ingestion

Pull weekly actual stats from FantasAI Cloudflare Worker API (Sleeper data source) and land it in bronze and silver Delta tables.

In [0]:
import requests
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

BASE_URL = "https://api.fantasai.net"

# Try to fetch current week and season from FantasAI API
try:
    response = requests.get(f"{BASE_URL}/api/v1/week/current", timeout=10)
    response.raise_for_status()
    current_data = response.json()
    
    WEEK = current_data.get('week')
    SEASON = current_data.get('season')
    WEEK_TYPE = current_data.get('type')  # regular, playoff, etc.
    
    print(f"✅ Fetched from API: Week {WEEK}, Season {SEASON} ({WEEK_TYPE})")
    
except Exception as e:
    # Fallback to hardcoded values if endpoint not available yet
    print(f"⚠ API endpoint not available yet: {e}")
    print("Using hardcoded values (update when API is live)")
    WEEK = 18
    SEASON = 2025
    WEEK_TYPE = "regular"
    print(f"📅 Using: Week {WEEK}, Season {SEASON} ({WEEK_TYPE})")

TOP_N = 100  # Number of top players to fetch

In [0]:
# Create bronze stats table
spark.sql("""
CREATE TABLE IF NOT EXISTS main.fantasai.bronze_weekly_stats (
  player_id STRING,
  week INT,
  season INT,
  fantasy_points DOUBLE,
  stats STRING,
  ingested_at TIMESTAMP
)
USING DELTA
""")

# Create silver stats table
spark.sql("""
CREATE TABLE IF NOT EXISTS main.fantasai.silver_weekly_stats (
  player_id STRING,
  week INT,
  season INT,
  fantasy_points DOUBLE,
  stats STRING,
  ingested_at TIMESTAMP
)
USING DELTA
""")

print("✓ Tables created")

In [0]:
# Fetch weekly stats from Cloudflare Worker API
import json

response = requests.get(
    f"{BASE_URL}/api/v1/stats/week",
    params={"week": WEEK, "season": SEASON, "top": TOP_N},
    timeout=30
)
response.raise_for_status()
response_data = response.json()

# Extract stats dictionary from nested response
stats_dict = response_data.get('stats', {})
print(f"Fetched stats for {len(stats_dict)} players")

# Convert to DataFrame rows - stats keyed by player_id
rows = []
for player_id, stat_data in stats_dict.items():
    # Extract fantasy points from stats
    fantasy_points = stat_data.get('pts_ppr', 0) or stat_data.get('pts_half_ppr', 0) or stat_data.get('pts_std', 0)
    
    rows.append(
        Row(
            player_id=str(player_id),
            week=WEEK,
            season=SEASON,
            fantasy_points=float(fantasy_points),
            stats=json.dumps(stat_data),  # Store all stats as JSON
        )
    )

stats_df = spark.createDataFrame(rows)

display(stats_df)

In [0]:
# Write to bronze table using MERGE to handle duplicates
bronze_df = stats_df.withColumn("ingested_at", F.current_timestamp()).withColumn("week", F.col("week").cast('int')).withColumn("season", F.col("season").cast('int'))

# Create temp view for merge
bronze_df.createOrReplaceTempView("bronze_updates")

# Perform MERGE operation
spark.sql("""
  MERGE INTO main.fantasai.bronze_weekly_stats AS target
  USING bronze_updates AS source
  ON target.player_id = source.player_id 
    AND target.week = source.week 
    AND target.season = source.season
  WHEN MATCHED THEN
    UPDATE SET
      target.fantasy_points = source.fantasy_points,
      target.stats = source.stats,
      target.ingested_at = source.ingested_at
  WHEN NOT MATCHED THEN
    INSERT (player_id, week, season, fantasy_points, stats, ingested_at)
    VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.ingested_at)
""")

print(f"✓ Merged {bronze_df.count()} records into bronze_weekly_stats (upsert)")

In [0]:
# Transform and deduplicate for silver (latest stats for player/week/season)
silver_df = (
    bronze_df
    .select(
        F.col("player_id").cast("string"),
        F.col("week").cast("int"),
        F.col("season").cast("int"),
        F.col("fantasy_points").cast("double"),
        F.col("stats").cast("string"),
        F.col("ingested_at"),
    )
    .dropDuplicates(["player_id", "week", "season"])
)

display(silver_df)

In [0]:
# Write to silver table using MERGE to handle duplicates
from delta.tables import DeltaTable

# Create temp view for merge
silver_df.createOrReplaceTempView("silver_updates")

# Perform MERGE operation
spark.sql("""
  MERGE INTO main.fantasai.silver_weekly_stats AS target
  USING silver_updates AS source
  ON target.player_id = source.player_id 
    AND target.week = source.week 
    AND target.season = source.season
  WHEN MATCHED THEN
    UPDATE SET
      target.fantasy_points = source.fantasy_points,
      target.stats = source.stats,
      target.ingested_at = source.ingested_at
  WHEN NOT MATCHED THEN
    INSERT (player_id, week, season, fantasy_points, stats, ingested_at)
    VALUES (source.player_id, source.week, source.season, source.fantasy_points, source.stats, source.ingested_at)
""")

print(f"✓ Merged {silver_df.count()} records into silver_weekly_stats (upsert)")

In [0]:
%sql
-- Check ingested data
SELECT 
  player_id,
  week,
  season,
  fantasy_points,
  ingested_at
FROM main.fantasai.silver_weekly_stats
WHERE week = 18 AND season = 2025
ORDER BY fantasy_points DESC
LIMIT 20

In [0]:
%sql
-- Get unique records (latest ingestion) for top performers
WITH latest_records AS (
  SELECT 
    player_id,
    week,
    season,
    fantasy_points,
    stats,
    ROW_NUMBER() OVER (PARTITION BY player_id, week, season ORDER BY ingested_at DESC) as rn
  FROM main.fantasai.silver_weekly_stats
  WHERE week = 18 AND season = 2025
)
SELECT 
  player_id,
  fantasy_points,
  get_json_object(stats, '$.rec') as receptions,
  get_json_object(stats, '$.rec_yd') as receiving_yards,
  get_json_object(stats, '$.rec_td') as receiving_tds,
  get_json_object(stats, '$.rush_yd') as rushing_yards,
  get_json_object(stats, '$.rush_td') as rushing_tds,
  get_json_object(stats, '$.pass_yd') as passing_yards,
  get_json_object(stats, '$.pass_td') as passing_tds
FROM latest_records
WHERE rn = 1
ORDER BY fantasy_points DESC
LIMIT 25

In [0]:
%sql
-- Check for duplicates and data quality
SELECT 
  'Total records' as metric,
  COUNT(*) as value
FROM main.fantasai.silver_weekly_stats
WHERE week = 18 AND season = 2025

UNION ALL

SELECT 
  'Unique player/week/season' as metric,
  COUNT(DISTINCT CONCAT(player_id, '-', week, '-', season)) as value
FROM main.fantasai.silver_weekly_stats
WHERE week = 18 AND season = 2025

UNION ALL

SELECT 
  'Ingestion runs' as metric,
  COUNT(DISTINCT ingested_at) as value
FROM main.fantasai.silver_weekly_stats
WHERE week = 18 AND season = 2025

In [0]:
%sql
-- Remove old duplicates, keeping only the latest record for each player/week/season
CREATE OR REPLACE TABLE main.fantasai.silver_weekly_stats AS
WITH ranked_records AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (PARTITION BY player_id, week, season ORDER BY ingested_at DESC) as rn
  FROM main.fantasai.silver_weekly_stats
)
SELECT 
  player_id,
  week,
  season,
  fantasy_points,
  stats,
  ingested_at
FROM ranked_records
WHERE rn = 1;

-- Also clean up bronze table
CREATE OR REPLACE TABLE main.fantasai.bronze_weekly_stats AS
WITH ranked_records AS (
  SELECT 
    *,
    ROW_NUMBER() OVER (PARTITION BY player_id, week, season ORDER BY ingested_at DESC) as rn
  FROM main.fantasai.bronze_weekly_stats
)
SELECT 
  player_id,
  week,
  season,
  fantasy_points,
  stats,
  ingested_at
FROM ranked_records
WHERE rn = 1;

SELECT 'Cleanup complete' as status, 
       COUNT(*) as records_in_silver
FROM main.fantasai.silver_weekly_stats